In [1]:
from collections import Counter

import osmium
import pandas as pd

from isochrone.download import RAW_DATA_DIR, osm_timestamp

OSM_PATH = RAW_DATA_DIR / "oberbayern_2026-11-05.osm.pbf"
print("data up to", osm_timestamp(OSM_PATH))

highway = Counter()
other = {key: Counter() for key in ["foot", "access", "sidewalk", "motorroad", "area", "indoor"]}
ways = osmium.FileProcessor(OSM_PATH, osmium.osm.WAY).with_filter(
    osmium.filter.KeyFilter("highway")
)
for way in ways:
    highway[way.tags["highway"]] += 1
    for key, counter in other.items():
        if key in way.tags:
            counter[way.tags[key]] += 1

for key, counter in other.items():
    print(key, counter.most_common(6))
print(sum(highway.values()), "ways with a highway tag,", len(highway), "values")
pd.Series(highway).sort_values(ascending=False)


data up to 2026-11-03T21:24:23Z
foot [('yes', 33881), ('designated', 32707), ('use_sidepath', 12263), ('no', 4370), ('permissive', 1383), ('private', 746)]
access [('private', 103023), ('no', 5082), ('permissive', 3879), ('yes', 3816), ('customers', 3431), ('destination', 2033)]
sidewalk [('no', 32554), ('both', 18356), ('right', 7358), ('separate', 6563), ('left', 5954), ('yes', 40)]
motorroad [('yes', 2140), ('no', 653)]
area [('yes', 3427), ('no', 40)]
indoor [('yes', 5896), ('no', 101), ('room', 47), ('corridor', 10), ('footway', 3), ('highway', 3)]
984636 ways with a highway tag, 41 values


service              285590
track                198096
footway              159335
path                 105518
residential          100367
unclassified          34269
secondary             23364
tertiary              22099
steps                 16516
primary               10158
living_street          6135
cycleway               4759
motorway               3740
bus_stop               2173
motorway_link          2139
pedestrian             1966
trunk                  1480
primary_link           1119
trunk_link             1095
corridor               1084
secondary_link          841
proposed                824
construction            553
platform                519
tertiary_link           302
bridleway               197
elevator                 96
raceway                  80
rest_area                70
via_ferrata              67
services                 29
busway                   15
road                     13
yes                      13
ladder                    5
disused             

In [2]:
import time

import numpy as np
from scipy.sparse.csgraph import connected_components

from isochrone.osm import build_walking_graph

start = time.perf_counter()
graph = build_walking_graph(OSM_PATH)
print(f"built in {time.perf_counter() - start:.0f} s")

n_nodes = len(graph.node_ids)
print(n_nodes, "nodes,", graph.lengths.nnz // 2, "edges")
print(f"total length {graph.lengths.sum() / 2 / 1000:.0f} km")
print("edge length (m), percentiles 50, 90, 99, 100:")
print(np.percentile(graph.lengths.data, [50, 90, 99, 100]).round(1))

n_components, component = connected_components(graph.lengths, directed=False)
sizes = np.sort(np.bincount(component))[::-1]
print(n_components, f"components; the largest has {sizes[0] / n_nodes:.1%} of the nodes")
print("sizes of the 10 largest:", sizes[:10])
print("components with at most 10 nodes:", (sizes <= 10).sum())


built in 12 s
5156527 nodes, 5450977 edges
total length 130456 km
edge length (m), percentiles 50, 90, 99, 100:
[  15.    50.4  148.6 1845.6]
6854 components; the largest has 98.9% of the nodes
sizes of the 10 largest: [5098093     982     731     643     584     408     311     308     292
     261]
components with at most 10 nodes: 5600
